In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_SuperZZ1_theta,MSE_SuperZZ1_theta,R2_SuperZZ2_theta,MSE_SuperZZ2_theta,...,R2_ZZx2_theta,MSE_ZZx2_theta,R2_ZZxReto_theta,MSE_ZZxReto_theta,R2_ZZy1_theta,MSE_ZZy1_theta,R2_ZZy2_theta,MSE_ZZy2_theta,R2_semiCirc_theta,MSE_semiCirc_theta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed2289,[1],0.3,0.7,0.01,2289,0.734311,0.543805,-0.361508,0.524084,...,-5.235106,0.133114,-0.274306,0.436998,-22.339933,0.052244,0.864460,0.552031,-13.230188,0.142813
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed1227,[1],0.3,0.7,0.01,1227,0.679990,0.491530,-1.132651,0.464763,...,-5.325339,0.026271,-1.141469,0.332208,-25.246598,-0.037455,0.807808,0.485788,-9.374910,0.191700
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed2484,[1],0.3,0.7,0.01,2484,0.787643,0.547311,-1.266310,0.515486,...,-4.414953,0.161809,-0.558644,0.446198,-23.656994,0.051241,0.782914,0.549305,-12.549780,0.154683
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed7740,[1],0.3,0.7,0.01,7740,0.670913,0.537957,-1.515975,0.519115,...,-4.085015,0.152571,-0.982513,0.433930,-25.539057,0.026834,0.757222,0.548813,-11.783941,0.160321
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed3838,[1],0.3,0.7,0.01,3838,0.719622,0.510133,-1.188948,0.474123,...,-5.469749,0.039823,-1.025990,0.356412,-27.412366,-0.041551,0.808443,0.497002,-11.658792,0.142671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2010,model_arch100_r0.9_Ld0.7_Lp0.3_seed4485,[100],0.7,0.3,0.90,4485,0.907804,0.629156,-1.165198,0.558621,...,-0.517044,0.454550,0.015158,0.471647,-35.932318,-0.042008,0.724130,0.585115,-18.143590,0.020954
2011,model_arch100_r0.9_Ld0.7_Lp0.3_seed1627,[100],0.7,0.3,0.90,1627,0.693828,0.700339,0.332334,0.561070,...,-0.089932,0.495318,-0.476961,0.385498,-21.216489,0.085453,0.778628,0.619261,-7.193388,0.366884
2012,model_arch100_r0.9_Ld0.7_Lp0.3_seed7931,[100],0.7,0.3,0.90,7931,0.883439,0.643459,-0.749872,0.553518,...,-0.051205,0.475382,-0.488485,0.379921,-36.878514,-0.129852,0.930722,0.593099,-16.747263,0.022675
2013,model_arch100_r0.9_Ld0.7_Lp0.3_seed8994,[100],0.7,0.3,0.90,8994,0.894703,0.638118,-1.095203,0.564140,...,-0.024202,0.466998,-0.232294,0.449245,-44.372972,-0.139715,0.705281,0.587055,-16.161246,0.053132


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "SuperZZ1":  "Train",
    "SuperZZ2":  "Val",
    "ZZx1":      "Test",
    "ZZx2":     "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
1414,model_arch71_r0.01_Ld0.7_Lp0.3_seed3838,[71],0.845034,0.300534,-2.275080,-0.733636
106,model_arch6_r0.9_Ld0.3_Lp0.7_seed1227,[6],0.776751,0.381390,-2.579595,-0.806721
1932,model_arch96_r0.9_Ld0.7_Lp0.3_seed7931,[96],0.827609,0.036386,-2.266306,-0.968500
949,model_arch48_r0.9_Ld0.3_Lp0.7_seed3838,[48],0.880464,-0.537318,-2.312674,-1.054228
1412,model_arch71_r0.01_Ld0.7_Lp0.3_seed2484,[71],0.779551,0.239566,-3.159034,-1.161803



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_SuperZZ1_theta,R2_SuperZZ2_theta,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
1414,model_arch71_r0.01_Ld0.7_Lp0.3_seed3838,[71],0.845034,0.300534,-1.795692,0.507814,-11.440731,0.826836,-2.892717,-0.855413,0.478459,-1.077587,-4.226695,0.845034,0.300534,-2.275080,-0.733636
106,model_arch6_r0.9_Ld0.3_Lp0.7_seed1227,[6],0.776751,0.381390,-6.217021,0.386179,0.000506,0.946560,-3.597019,-0.870064,0.518540,-6.758858,-7.625174,0.776751,0.381390,-2.579595,-0.806721
1932,model_arch96_r0.9_Ld0.7_Lp0.3_seed7931,[96],0.827609,0.036386,-0.581263,0.609595,-16.761775,0.938633,-1.888318,-0.281786,-0.593544,-0.043324,-1.794972,0.827609,0.036386,-2.266306,-0.968500
949,model_arch48_r0.9_Ld0.3_Lp0.7_seed3838,[48],0.880464,-0.537318,-1.315104,0.173144,-9.656484,0.697972,-1.085931,0.163117,0.674955,-0.305597,-10.160142,0.880464,-0.537318,-2.312674,-1.054228
1412,model_arch71_r0.01_Ld0.7_Lp0.3_seed2484,[71],0.779551,0.239566,-2.450593,0.303169,-14.207218,0.707050,-3.690387,-1.253140,0.765231,-1.656752,-6.948666,0.779551,0.239566,-3.159034,-1.161803


In [5]:
final_table.to_excel("BestModels-1l.xlsx")